<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/183-Mini-Project-M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai langchain langchain_openai langchain_community langchain_chroma langchain_text_splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 

In [2]:
from google.colab import userdata
import openai
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
openai.api_key  = os.environ["OPENAI_API_KEY"]

1. Sample texts:

texts 리스트에는 샘플 텍스트들이 포함되어 있습니다. 각 텍스트는 광대버섯(Amanita phalloides)에 대한 정보를 제공합니다.
Custom text loader function:

2. Custom text loader function:

load_texts 함수는 주어진 텍스트 리스트를 받아서 Document 객체의 리스트로 변환합니다. Document 객체는 페이지 콘텐츠(텍스트 내용)와 메타데이터를 포함합니다.

3.Load the documents:

load_texts 함수를 사용하여 raw_documents 변수에 문서를 로드합니다.

4. Initialize the text splitter and embeddings:

CharacterTextSplitter 객체와 OpenAIEmbeddings 객체를 초기화합니다. CharacterTextSplitter는 텍스트를 일정 크기의 조각으로 분할하고, OpenAIEmbeddings는 텍스트 조각을 임베딩 벡터로 변환하는 역할을 합니다.

5. Split the documents into chunks:

각 문서를 100자 크기의 텍스트 조각으로 분할합니다. chunk_overlap이 0으로 설정되어 있으므로, 조각 사이에 중복이 없습니다.

6. Embed each chunk:

각 텍스트 조각을 임베딩합니다. 임베딩은 텍스트 조각을 벡터로 변환하여 기계 학습 모델이 이해할 수 있도록 합니다.

7. Initialize the vector store with embedded chunks:

Chroma 벡터 스토어를 초기화하고, 임베딩된 텍스트 조각들을 저장합니다. 이 벡터 스토어는 나중에 유사한 텍스트 검색을 수행하는 데 사용됩니다.

In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document  # Import the Document class

# Sample texts
texts = [
    "광대버섯(Amanita phalloides)은 크고 인상적인 후성(위) 자실체(담자과체)를 가지고 있습니다.",
    "큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.",
    "A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.",
    "고양이 다리의 갯수는 3개입니다",
]


In [12]:

# Custom text loader function
def load_texts(text_list):
    documents = []
    for text in text_list:
        documents.append(Document(page_content=text, metadata={}))  # Create Document objects
    return documents


In [13]:

# Load the documents
raw_documents = load_texts(texts)

# Initialize the text splitter and embeddings
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
embeddings = OpenAIEmbeddings()

# Split the documents into chunks
chunks = []
for doc in raw_documents:
    chunks.extend(text_splitter.split_text(doc.page_content))

# Embed each chunk
embedded_chunks = embeddings.embed_documents([chunk for chunk in chunks])  # Pass list of strings

# Initialize the vector store with embedded chunks
vector_store = Chroma.from_documents(
    [Document(page_content=chunk, metadata={}) for chunk in chunks],
    embeddings
)

In [ ]:
embedded_chunks

In [14]:
def dynamic_search(question):
    try:
        k = int(input("원하는 검색 결과의 수를 입력하세요 (1-3): "))
        if k < 1 or k > 3:
            raise ValueError("검색 결과의 수는 1에서 3 사이여야 합니다.")
    except ValueError as e:
        print(f"잘못된 입력입니다: {e}")
        return []

    # Ensure vector_store is properly initialized and accessible
    return vector_store.similarity_search_with_score(query=question, k=k)

In [9]:
# 사용자 질문에 따른 검색
question = "큰 자실체를 가진 순백색 버섯에 대해 알려주세요"
search_results = dynamic_search(question)
print("검색 결과:", search_results)

원하는 검색 결과의 수를 입력하세요 (1-3): 3
검색 결과: [(Document(id='977df41c-ab36-4dfb-bb1c-cf1a95ccc12d', metadata={}, page_content='큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.'), 0.20695549249649048), (Document(id='78bd1f45-9dbc-40b0-85dd-d400ce4090fb', metadata={}, page_content='A. phalloides, 일명 Death Cap은 알려진 모든 버섯 중에서 가장 독성이 강한 버섯 중 하나입니다.'), 0.31642743945121765), (Document(id='568435f0-0619-4847-a12d-f20b494312cf', metadata={}, page_content='광대버섯(Amanita phalloides)은 크고 인상적인 후성(위) 자실체(담자과체)를 가지고 있습니다.'), 0.3177797198295593)]


In [16]:
# 사용자 질문에 따른 검색
question = "고양이 다리는 몇개인가요?"
search_results = dynamic_search(question)
print("검색 결과:", search_results)

원하는 검색 결과의 수를 입력하세요 (1-3): 3
검색 결과: [(Document(id='3f6569e3-63ff-47e2-9904-4faad10e5c2b', metadata={}, page_content='고양이 다리의 갯수는 3개입니다'), 0.1272822767496109), (Document(id='977df41c-ab36-4dfb-bb1c-cf1a95ccc12d', metadata={}, page_content='큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.'), 0.3862050771713257), (Document(id='4ccb1d5b-130a-462b-9d9a-07931a9e85bc', metadata={}, page_content='큰 자실체를 가진 버섯은 Amanita phalloides입니다. 일부 품종은 모두 흰색입니다.'), 0.3862050771713257)]


In [18]:
from langchain_openai import OpenAIEmbeddings

embeddings_instance = OpenAIEmbeddings(model='text-embedding-3-small')
default_model = embeddings_instance.model
print(f"OpenAIEmbeddings()의 기본 임베딩 모델은 '{default_model}' 입니다.")

OpenAIEmbeddings()의 기본 임베딩 모델은 'text-embedding-3-small' 입니다.
